# 🏭 AHU Analysis
**Purpose:** Analyse the central Air Handling Unit performance —  
cooling/heating coil behaviour, fan power, outdoor air mixing, and energy consumption.

In [ ]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from data_wrangling import DataInspector, DataPlotter

CSV_PATH = './results/state_log.csv'
df = pd.read_csv(CSV_PATH)

# Build datetime index (EnergyPlus uses Hour=24 for midnight → roll to next day)
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)

ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps')

## 1 · Temperature Through AHU Stages
Outdoor → Mixed Air → Cooling Coil → Heating Coil → Fan → Supply

In [ ]:
fig = go.Figure()
t = df['Datetime']

stages = [
    ('Outdoor_Air_Temp_C',  'Outdoor Air',   '#636EFA'),
    ('Mixed_Air_Temp_C',    'Mixed Air',     '#EF553B'),
    ('CC_Out_Temp_C',       'After CC',      '#00CC96'),
    ('HC_Out_Temp_C',       'After HC',      '#FFA15A'),
    ('Fan_Out_Temp_C',      'Fan Out (Supply)', '#AB63FA'),
]

for col, name, color in stages:
    fig.add_trace(go.Scatter(x=t, y=df[col], name=name,
        line=dict(color=color, width=1.8)))

fig.update_layout(template='plotly_dark', height=450,
    title='AHU Temperature Cascade',
    yaxis_title='°C',
    legend=dict(orientation='h', y=-0.12))
fig.show()

## 2 · Humidity Through AHU Stages

In [ ]:
fig = go.Figure()
t = df['Datetime']

stages_rh = [
    ('Outdoor_Air_RH_pct', 'Outdoor Air',    '#636EFA'),
    ('Mixed_Air_RH_pct',   'Mixed Air',      '#EF553B'),
    ('CC_Out_RH_pct',      'After CC',       '#00CC96'),
    ('HC_Out_RH_pct',      'After HC',       '#FFA15A'),
    ('Fan_Out_RH_pct',     'Fan Out (Supply)', '#AB63FA'),
]

for col, name, color in stages_rh:
    fig.add_trace(go.Scatter(x=t, y=df[col], name=name,
        line=dict(color=color, width=1.8)))

fig.update_layout(template='plotly_dark', height=450,
    title='AHU Relative Humidity Cascade',
    yaxis_title='RH %',
    legend=dict(orientation='h', y=-0.12))
fig.show()

## 3 · Central Equipment Energy Consumption

In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['Cooling Coil Power', 'Heating Coil Power', 'Fan Power'],
    vertical_spacing=0.08
)
t = df['Datetime']

fig.add_trace(go.Scatter(x=t, y=df['CC_Power_W'], name='Cooling Coil',
    fill='tozeroy', line=dict(color='#00CC96', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=df['HC_Power_W'], name='Heating Coil',
    fill='tozeroy', line=dict(color='#EF553B', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=df['Fan_Power_W'], name='Fan',
    fill='tozeroy', line=dict(color='#AB63FA', width=1.5)), row=3, col=1)

fig.update_layout(template='plotly_dark', height=600,
    title='Central Plant Energy Consumption',
    legend=dict(orientation='h', y=-0.06))
fig.update_yaxes(title_text='W', row=1, col=1)
fig.update_yaxes(title_text='W', row=2, col=1)
fig.update_yaxes(title_text='W', row=3, col=1)
fig.show()

## 4 · Mass Flow Distribution Across Zones

In [ ]:
fig = go.Figure()
t = df['Datetime']
colors = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA']

zones = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
for z, c in zip(zones, colors):
    fig.add_trace(go.Scatter(x=t, y=df[f'{z}_VAV_Flow_kg_s'],
        name=z, line=dict(color=c, width=1.5), stackgroup='flow'))

fig.add_trace(go.Scatter(x=t, y=df['Fan_Out_Flow_kg_s'],
    name='Total Fan Out', line=dict(color='white', width=2, dash='dot')))

fig.update_layout(template='plotly_dark', height=450,
    title='VAV Flow Distribution (Stacked) vs Total Fan Output',
    yaxis_title='kg/s',
    legend=dict(orientation='h', y=-0.12))
fig.show()

## 5 · CO₂ Through AHU — Outdoor → Mixed → Supply

In [ ]:
fig = go.Figure()
t = df['Datetime']

co2_stages = [
    ('Outdoor_Air_CO2_ppm', 'Outdoor Air',     '#636EFA'),
    ('Mixer_Inlet_CO2_ppm', 'Return Air',      '#EF553B'),
    ('Mixed_Air_CO2_ppm',   'Mixed Air',       '#00CC96'),
    ('Fan_Out_CO2_ppm',     'Supply (Fan Out)', '#AB63FA'),
]

for col, name, color in co2_stages:
    fig.add_trace(go.Scatter(x=t, y=df[col], name=name,
        line=dict(color=color, width=1.8)))

fig.update_layout(template='plotly_dark', height=450,
    title='CO₂ Concentration Through AHU',
    yaxis_title='ppm',
    legend=dict(orientation='h', y=-0.12))
fig.show()

## 6 · Reheater Power Per Zone

In [ ]:
fig = go.Figure()
t = df['Datetime']
colors = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA']
zones = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']

for z, c in zip(zones, colors):
    fig.add_trace(go.Scatter(x=t, y=df[f'{z}_Reheater_W'],
        name=z, line=dict(color=c, width=1.5)))

fig.update_layout(template='plotly_dark', height=450,
    title='Zone Reheater Power',
    yaxis_title='W',
    legend=dict(orientation='h', y=-0.12))
fig.show()

## 7 · Energy Statistics (data_wrangling)

In [ ]:
energy_cols = ['CC_Power_W', 'HC_Power_W', 'Fan_Power_W']
zones = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
energy_cols += [f'{z}_Reheater_W' for z in zones]

di = DataInspector()
di.df = df[energy_cols].copy()
di.get_summary(detailed=True)

In [ ]:
# Energy distribution
di.summary_plot(
    columns=['CC_Power_W', 'HC_Power_W', 'Fan_Power_W'],
    numeric_plots=['violin', 'histogram'],
    separate_plots=True
)

In [ ]:
# Total energy consumed (Wh) over simulation period
dt_hours = 5 / 60  # 5-minute timesteps
print('=== Total Energy (kWh) ===')
for col in ['CC_Power_W', 'HC_Power_W', 'Fan_Power_W']:
    total_kwh = (df[col].sum() * dt_hours) / 1000
    print(f'  {col:20s}: {total_kwh:10.2f} kWh')

print('\n=== Zone Reheater Energy (kWh) ===')
for z in zones:
    col = f'{z}_Reheater_W'
    total_kwh = (df[col].sum() * dt_hours) / 1000
    print(f'  {z:15s}: {total_kwh:10.2f} kWh')